# Apex Capital — M&A Multi-Agent System
## Complete Runbook

This notebook contains all commands to run every phase of the project.
Run cells in order, or skip to any phase.

**Prerequisite:** Set your DeepSeek API key in the `.env` file:
```
DEEPSEEK_API_KEY=sk-your-key-here
```

In [ ]:
import os, sys, json
from datetime import datetime

BASE_DIR = os.getcwd()
print(f'Working directory: {BASE_DIR}')
print(f'Python: {sys.version}')

---
## Phase 1 — Single-Agent Baseline
Deterministic benchmark. No LLM. One monolithic agent does screening, financial analysis, risk assessment, strategic fit, and scoring.

In [ ]:
print('=== PHASE 1: Single-Agent Baseline ===')
sys.path.insert(0, os.path.join(BASE_DIR, 'single_agent'))
from baseline import SingleAgentBaseline
agent = SingleAgentBaseline()
result = agent.run()

In [ ]:
print('=== PHASE 1: Validation Tests ===')
# Run the test functions directly (works in notebooks and CLI)
sys.path.insert(0, BASE_DIR)
from test_phase1 import test_data_exists, test_logs_generated, test_screening_logic
for t in [test_data_exists, test_logs_generated, test_screening_logic]:
    try:
        t()
        print(f'PASS: {t.__name__}')
    except Exception as e:
        print(f'FAIL: {t.__name__} — {e}')

---
## Phase 2 — Multi-Agent Orchestrator
Real DeepSeek LLM-powered agents. Orchestrator coordinates Scout → Financial Analyst → Risk Analyst → Strategy Analyst → Report Writer. Fan-out runs financial/risk/strategy analysts in parallel per company.

In [ ]:
print('=== PHASE 2: Multi-Agent Orchestrator ===')
sys.path.insert(0, os.path.join(BASE_DIR, 'multi_agent'))
from orchestrator import Orchestrator
orch = Orchestrator()
report = orch.run()

---
## Phase 3 — Fan-Out Optimizer
Benchmarks sequential vs parallel execution. Measures speedup by comparing sequential analysis (one analyst at a time) vs fan-out (all analysts in parallel).

In [ ]:
print('=== PHASE 3: Fan-Out Benchmark ===')
sys.path.insert(0, os.path.join(BASE_DIR, 'multi_agent'))
from fanout import FanOutBenchmark
bm = FanOutBenchmark()
result = bm.run_benchmark()

---
## Phase 4 — Evaluator Agent
Meta-evaluates the entire architecture. Scores 1-10 on:
- Architecture design (30%)
- Prompt quality (25%)
- Output quality (25%)
- Scalability & robustness (20%)

In [ ]:
print('=== PHASE 4: Evaluator Agent ===')
sys.path.insert(0, os.path.join(BASE_DIR, 'evaluator'))
from evaluator import run_evaluator
evaluation = run_evaluator()

---
## Phase 5 — Improvement Loop
Closed feedback loop:
1. Loads evaluator scores
2. Extracts top improvements
3. LLM rewrites weakest agent prompts
4. Promotes improved prompts as active
5. Re-evaluates and shows before/after delta

In [ ]:
print('=== PHASE 5: Improvement Loop ===')
sys.path.insert(0, os.path.join(BASE_DIR, 'evaluator'))
from improvement_loop import run_improvement_loop
result = run_improvement_loop()

---
## Phase 6 — Dashboard
Two options:
1. **Static HTML** — self-contained dashboard
2. **Interactive Server** — Flask app with live pipeline controls

In [ ]:
print('=== PHASE 6: Generate Static Dashboard ===')
sys.path.insert(0, os.path.join(BASE_DIR, 'dashboard'))
from generate_dashboard import generate
generate()
print('Open dashboard/index.html in your browser')

In [ ]:
# Uncomment to start the interactive Flask server (blocks execution):
# !python dashboard/server.py
# Then open http://localhost:5000

---
## Phase 7 — Log Analyzer
Parses all JSON logs across every phase. Produces timing stats, score distributions, evaluator progression, improvement deltas, and cross-phase comparison.

In [ ]:
print('=== PHASE 7: Log Analyzer ===')
sys.path.insert(0, BASE_DIR)
from log_analyzer import analyze
analyze()

---
## Inspect Results

In [ ]:
print('=== Multi-Agent Report ===')
report_path = os.path.join(BASE_DIR, 'logs', 'report_multi_agent.json')
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        data = json.load(f)
    print(f"Passed screening: {data.get('passed_screening', 'N/A')}")
    for c in data.get('ranked_companies', [])[:5]:
        print(f"  {c['rank']}. {c['name']} — {c['rating']} ({c['composite_score']}) | F:{c['financial_score']} R:{c['risk_score']} S:{c['strategy_score']}")
else:
    print('No report yet. Run Phase 2 first.')

In [ ]:
print('=== Evaluator Scores ===')
scores_path = os.path.join(BASE_DIR, 'evaluator', 'scores.json')
if os.path.exists(scores_path):
    with open(scores_path, 'r') as f:
        data = json.load(f)
    for i, it in enumerate(data.get('iterations', []), 1):
        e = it['evaluation']
        print(f"  Iteration {i}: {e.get('overall_score', 'N/A')}/10 — {e.get('production_readiness', 'N/A')}")
else:
    print('No evaluation yet. Run Phase 4 first.')

In [ ]:
print('=== Fan-Out Benchmark ===')
bm_path = os.path.join(BASE_DIR, 'logs', 'run_3_fanout_benchmark.json')
if os.path.exists(bm_path):
    with open(bm_path, 'r') as f:
        data = json.load(f)
    t = data.get('timings', {})
    print(f"  Sequential: {t.get('sequential_analysis_sec', 'N/A')}s")
    print(f"  Fan-out:    {t.get('fanout_analysis_sec', 'N/A')}s")
    print(f"  Speedup:    {t.get('speedup_x', 'N/A')}x")
    print(f"  Reduction:  {t.get('time_reduction_pct', 'N/A')}%")
else:
    print('No benchmark yet. Run Phase 3 first.')

In [ ]:
print('=== Created Files ===')
for root, dirs, files in os.walk(BASE_DIR):
    for f in files:
        if any(root.startswith(os.path.join(BASE_DIR, d)) for d in ['logs', '__pycache__', '.git', 'venv']):
            continue
        if f.endswith('.pyc'):
            continue
        rp = os.path.relpath(os.path.join(root, f), BASE_DIR)
        print(f'  {rp}')

---
## Clear All Outputs / Reset
Run this cell to clear all logs and agent memory (preserves source code and prompts).

In [ ]:
import shutil

# Clear logs
logs_dir = os.path.join(BASE_DIR, 'logs')
if os.path.exists(logs_dir):
    for f in os.listdir(logs_dir):
        os.remove(os.path.join(logs_dir, f))
    print('Cleared logs/')

# Clear agent memory
agents_dir = os.path.join(BASE_DIR, 'multi_agent', 'agents')
for agent in os.listdir(agents_dir):
    mem_dir = os.path.join(agents_dir, agent, 'memory')
    if os.path.isdir(mem_dir):
        for f in os.listdir(mem_dir):
            if f.endswith('.json'):
                os.remove(os.path.join(mem_dir, f))
        print(f'Cleared multi_agent/agents/{agent}/memory/')

# Clear evaluator scores
scores_path = os.path.join(BASE_DIR, 'evaluator', 'scores.json')
if os.path.exists(scores_path):
    os.remove(scores_path)
    print('Cleared evaluator/scores.json')

# Remove prompt backups (restore from prompt_v1.md if needed)
for agent in os.listdir(agents_dir):
    v1 = os.path.join(agents_dir, agent, 'prompt_v1.md')
    v2 = os.path.join(agents_dir, agent, 'prompt_v2.md')
    if os.path.exists(v1):
        os.rename(v1, os.path.join(agents_dir, agent, 'prompt.md'))
        print(f'Restored {agent}/prompt.md from backup')
    if os.path.exists(v2):
        os.remove(v2)
        print(f'Removed {agent}/prompt_v2.md')

print('Reset complete.')